In [29]:
import requests
import json
from pathlib import Path

In [ ]:
def extract_data(url_start, file_name, hasfields=True):
    # Extract data from the API
    url = url_start

    attribute_data_url = []
    attribute_data = []

    while url:  # Continue while there is a next page
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            if hasfields:
                attribute_data_url.extend(item["href"] for item in data["content"]["fields"])
            else:
                attribute_data_url.extend(item["href"] for item in data["content"])
            url = data.get("pageable", {}).get("nextPage")  # Update URL to the next page
        else:
            print(f"Error: {response.status_code}")
            break

    for url_data in attribute_data_url:
        response = requests.get(url_data)
        if response.status_code == 200:
            data = response.json()
            attribute_data.append(data)
        else:
            print(f"Error: {response.status_code} for URL: {url_data}")

    try:
        output_path = Path(f"data/raw/{file_name}.json")
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with output_path.open("w", encoding="utf-8") as f:
            json.dump(attribute_data, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
extract_data("https://digi-api.com/api/v1/attribute", "attributes")
extract_data("https://digi-api.com/api/v1/field", "fields")
extract_data("https://digi-api.com/api/v1/level", "levels")
extract_data("https://digi-api.com/api/v1/skill", "skills")
extract_data("https://digi-api.com/api/v1/digimon", "digimons", False)
